Here's a **step-by-step plan** for building a **Generative AI-powered Order Fulfillment Time Prediction system** for **Costco Wholesale**, using **3,000 dynamic order data entries** and a **free LLM model (like OpenAI GPT-4o-mini, LLaMA 2, or Falcon)**.

---

## **1️⃣ Problem Definition**

* **Objective:** Predict the **order fulfillment time** (in hours or days) based on multiple order-related factors using a mix of **ML + LLM (Generative AI)** for **better insights and explanations**.
* **Business Goal:** Improve supply chain efficiency, reduce delays, and provide **real-time delivery estimates** for customers.
* **Data Size:** 3,000+ records (can scale dynamically).

---

## **2️⃣ Data Collection & Features**

Generate or collect **synthetic data** (3,000 records) with features like:

| Feature Name                   | Type                 | Example Value        |
| ------------------------------ | -------------------- | -------------------- |
| Order\_ID                      | Categorical          | ORD-2025-001         |
| Product\_Category              | Categorical          | Electronics, Grocery |
| Order\_Quantity                | Numeric              | 12                   |
| Order\_Weight (kg)             | Numeric              | 5.5                  |
| Order\_Priority                | Categorical          | High, Medium, Low    |
| Payment\_Method                | Categorical          | Credit Card, COD     |
| Warehouse\_Location            | Categorical          | Seattle, Chicago     |
| Distance\_to\_Customer (km)    | Numeric              | 220                  |
| Carrier\_Type                  | Categorical          | Air, Road, Sea       |
| Weather\_Conditions            | Categorical          | Clear, Rain, Snow    |
| Holiday\_Season                | Binary               | 0 or 1               |
| Past\_Delivery\_Delay          | Numeric              | 2 (days)             |
| **Fulfillment\_Time (Target)** | Numeric (hours/days) | 36 hours             |

---

## **3️⃣ Tech Stack**

* **Language:** Python (FastAPI/Flask for API)
* **Database:** SQLite or PostgreSQL
* **ML Models:** RandomForestRegressor, XGBoost
* **Generative AI (Free LLM):**

  * **Option 1:** OpenAI GPT-4o-mini (free tier)
  * **Option 2:** Hugging Face free model (Falcon-7B, LLaMA-2-7B)
* **Libraries:** pandas, scikit-learn, matplotlib, joblib, transformers

## **4️⃣ Data Generation (Dynamic 3000 Records)**

In [1]:
import pandas as pd
import numpy as np
import random

categories = ['Electronics', 'Grocery', 'Clothing', 'Furniture']
priority = ['High', 'Medium', 'Low']
carriers = ['Air', 'Road', 'Sea']
weather = ['Clear', 'Rain', 'Snow']
locations = ['Seattle', 'Chicago', 'Dallas', 'San Francisco']

data = []
for i in range(3000):
    qty = random.randint(1, 100)
    weight = round(qty * random.uniform(0.2, 2.5), 2)
    dist = random.randint(10, 2000)
    base_time = dist / random.uniform(40, 80)
    priority_factor = 0.8 if random.choice(priority) == 'High' else 1.0
    weather_factor = 1.2 if random.choice(weather) != 'Clear' else 1.0
    fulfillment_time = round(base_time * priority_factor * weather_factor, 2)
    data.append([f"ORD-{i+1}", random.choice(categories), qty, weight, 
                 random.choice(priority), random.choice(['Credit Card','COD']), 
                 random.choice(locations), dist, random.choice(carriers),
                 random.choice(weather), random.choice([0,1]),
                 random.randint(0,5), fulfillment_time])

df = pd.DataFrame(data, columns=[
    "Order_ID","Product_Category","Order_Quantity","Order_Weight",
    "Order_Priority","Payment_Method","Warehouse_Location",
    "Distance_to_Customer","Carrier_Type","Weather_Conditions",
    "Holiday_Season","Past_Delivery_Delay","Fulfillment_Time"
])
df.to_csv("costco_orders.csv", index=False)


## **5️⃣ Build Machine Learning Model**

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,root_mean_squared_error

# Features (X) and Target (y)
X = df.drop(["Order_ID", "Fulfillment_Time"], axis=1)
y = df["Fulfillment_Time"]

# Encode categorical columns safely
le = LabelEncoder()
for col in X.select_dtypes(include=['object']).columns:
    X[col] = le.fit_transform(X[col].astype(str))  # ensure all are strings

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)


MAE: 3.9550796666666654
MSE: 31.687389540299996
RMSE: 5.629155313215296
R²: 0.7926855816794496


In [3]:
!pip3 install xgboost


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip


In [4]:
!pip3 install lightgbm


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip


In [5]:
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# from xgboost import XGBRegressor
# from lightgbm import LGBMRegressor
# import numpy as np

# # ------------------------
# # Features (X) and Target (y)
# # ------------------------
# X = df.drop(["Order_ID", "Fulfillment_Time"], axis=1)
# y = df["Fulfillment_Time"]

# # Identify categorical and numeric columns
# categorical = X.select_dtypes(include=['object']).columns
# numeric = X.select_dtypes(exclude=['object']).columns

# # Preprocessing: OneHotEncode categorical, keep numeric as is
# preprocessor = ColumnTransformer(
#     transformers=[
#         ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
#         ("num", "passthrough", numeric)
#     ]
# )

# # ------------------------
# # Train-test split
# # ------------------------
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# # ------------------------
# # Define models
# # ------------------------
# models = {
#     "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42),
#     "XGBoost": XGBRegressor(
#         n_estimators=500, learning_rate=0.05, max_depth=6,
#         subsample=0.8, colsample_bytree=0.8, random_state=42
#     ),
#     "LightGBM": LGBMRegressor(
#         n_estimators=500, learning_rate=0.05, max_depth=-1,
#         subsample=0.8, colsample_bytree=0.8, random_state=42
#     )
# }

# # ------------------------
# # Train, predict, evaluate
# # ------------------------
# results = {}

# for name, model in models.items():
#     pipeline = Pipeline(steps=[
#         ("preprocessor", preprocessor),
#         ("model", model)
#     ])
    
#     pipeline.fit(X_train, y_train)
#     y_pred = pipeline.predict(X_test)
    
#     mae = mean_absolute_error(y_test, y_pred)
#     mse = mean_squared_error(y_test, y_pred)
#     rmse = np.sqrt(mse)
#     r2 = r2_score(y_test, y_pred)
    
#     results[name] = {"MAE": mae, "MSE": mse, "RMSE": rmse, "R²": r2}

# # ------------------------
# # Print comparison
# # ------------------------
# print("\n--- Model Comparison ---")
# for model, metrics in results.items():
#     print(f"\n{model}:")
#     for metric, value in metrics.items():
#         print(f"{metric}: {value:.4f}")


In [ ]:
from transformers import pipeline

gen_model = pipeline("text-generation", model="tiiuae/falcon-7b-instruct")

def explain_prediction(input_data, predicted_time):
    prompt = f"""
    You are an AI assistant for Costco Order Management System.
    Given this order data: {input_data},
    The predicted fulfillment time is {predicted_time} hours.
    Explain to a customer why this delivery time was predicted.
    """
    result = gen_model(prompt, max_length=100)
    return result[0]['generated_text']

sample_input = X_test.iloc[0].to_dict()
explanation = explain_prediction(sample_input, round(y_pred[0], 2))
print(explanation)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]